In [1]:
import random

import boto3
import numpy as np
import pytz

from Data_query.trino_config import *

session = boto3.Session()
s3 = boto3.client("s3")

In [7]:
stop_trino()

Trino service stopping triggered.


In [3]:
sleep(120)

In [4]:
big_workers = 1
workers = 0
num_workers = max(workers, big_workers)
ensure_trino_running(worker_desired_count=workers, big_worker_desired_count=big_workers)
sleep(60)  # wait for trino to be ready

Trino service is not running. Starting the service...
Trino service triggered.
Service trino-service is now stable.


In [4]:
iceberg_exec("DROP TABLE IF EXISTS conformance_voltwattghi")
iceberg_exec("""CREATE TABLE conformance_voltwattghi (
                year BIGINT,
                month BIGINT,
                day BIGINT,
                site_id BIGINT,
                P_kW_sum DOUBLE,
                nonconformance_voltwattghi_sum DOUBLE,
                curtailment_voltwattghi_sum DOUBLE,
                nonconformance_voltwattghi_count BIGINT,
                curtailment_voltwattghi_count BIGINT,
                total_count BIGINT
            )
    """)

Executed
Executed


In [5]:
sleep(10)

In [6]:
v1 = 253
v2 = 260


def run_func(args):
    year, month, split_cons = args
    df = iceberg_sql(f"""
                insert into conformance_voltwattghi
                with data as (
                    select site_id, t_stamp, sum(power*circuit_polarity/1000) as P_kW, 
                            avg(voltage) as V, max(ac_capacity_kw) as ac_capacity_kw
                    from 
                    (select circuit_id, t_stamp, power, voltage 
                        from ts where year={year} and month={month} and is_pv=True and voltage > 0 and voltage < 300 and {split_cons}) as ts1
                inner join (select distinct site_id, circuit_id, circuit_polarity, ac_capacity_kw from meta_up23c where is_pv = True) as m on ts1.circuit_id = m.circuit_id
                group by site_id, t_stamp
                having avg(voltage) > 253),
                pq as (
                    select d.site_id, d.t_stamp, P_kW, d.V, ac_capacity_kw, vv.uncurtailed_P, vv.V as V_ghi,
                    (case when d.V < {v1} then ac_capacity_kw 
                    when d.V > {v2} then .2 * ac_capacity_kw
                    else ((ac_capacity_kw - .2 * ac_capacity_kw) / ({v1} - {v2}) * (d.V - {v2}) + .2 * ac_capacity_kw) end) + 0.04*ac_capacity_kw as max_P_volt_watt
                from data d left join voltwatt_uncurtailedPV as vv on d.site_id = vv.site_id and d.t_stamp = vv.t_stamp
                ),
                pq2 as (select site_id, t_stamp, day(t_stamp) as day, month(t_stamp) as month, year(t_stamp) as year, P_kW, uncurtailed_P, V, V_ghi, ac_capacity_kw, max_P_volt_watt,
                case when uncurtailed_P > max_P_volt_watt or uncurtailed_P is null then greatest(0, P_kW - max_P_volt_watt) else null end as nonconformance_voltwattghi,
                case when uncurtailed_P > max_P_volt_watt and P_kW < max_P_volt_watt then uncurtailed_P - P_kW else null end as curtailment_voltwattghi
                from pq 
                )
                
                select year, month, day, site_id, 
                sum(p_kW) as P_kW_sum,
                sum(nonconformance_voltwattghi) as nonconformance_voltwattghi_sum,
                sum(curtailment_voltwattghi) as curtailment_voltwattghi_sum,
                sum(case when nonconformance_voltwattghi > 0 then 1 else 0 end) as nonconformance_voltwattghi_count,
                sum(case when curtailment_voltwattghi > 0 then 1 else 0 end) as curtailment_voltwattghi_count,
                count(nonconformance_voltwattghi) as total_count
                from pq2
                group by year, month, day, site_id
                order by nonconformance_voltwattghi_sum desc
                """)
    # sleep(random.randint(1, 10))  # add some randomness to avoid overwhelming trino with simultaneous queries
    print(
        f"Completed year={year}, month={month},  {split_cons.replace('system.bucket(postcode, 16)', 'bucket')}"
    )
    return df


tasks = [
    (year, month, split_cons)
    for year in (2024,)
    for month in range(1, 13)
    for split_cons in [
        "system.bucket(postcode, 16) <= 7",
        "system.bucket(postcode, 16) > 7",
    ]
]
df = trino_parallel_batch(
    run_func, tasks, num_workers=num_workers, batch_size=num_workers
)
df

Completed year=2024, month=1,  bucket <= 7
Completed year=2024, month=1,  bucket > 7
Completed year=2024, month=2,  bucket <= 7
Completed year=2024, month=2,  bucket > 7
Completed year=2024, month=3,  bucket <= 7
Completed year=2024, month=3,  bucket > 7
Completed year=2024, month=4,  bucket <= 7
Completed year=2024, month=4,  bucket > 7
Completed year=2024, month=5,  bucket <= 7
Completed year=2024, month=5,  bucket > 7
Sleeping for 30 seconds to reduce load on Trino...
Completed year=2024, month=6,  bucket <= 7
Completed year=2024, month=6,  bucket > 7
Completed year=2024, month=7,  bucket <= 7
Completed year=2024, month=7,  bucket > 7
Completed year=2024, month=8,  bucket <= 7
Completed year=2024, month=8,  bucket > 7
Completed year=2024, month=9,  bucket <= 7
Completed year=2024, month=9,  bucket > 7
Completed year=2024, month=10,  bucket <= 7
Completed year=2024, month=10,  bucket > 7
Sleeping for 30 seconds to reduce load on Trino...
Completed year=2024, month=11,  bucket <= 7
Co

,rows
0,7061
1,8222
2,7123
3,8670
4,8042
5,10116
6,8048
7,10558
8,6826
9,8517


In [ ]:
iceberg_sql("""SELECT * FROM conformance_voltwattghi
            where uncurtailed_P """)

In [5]:
v1 = 253
v2 = 260


def run_func(args):
    year, month, split_cons = args
    df = iceberg_sql(f"""
                with data as (
                    select site_id, t_stamp, sum(power*circuit_polarity/1000) as P_kW, 
                            avg(voltage) as V, max(ac_capacity_kw) as ac_capacity_kw
                    from 
                    (select circuit_id, t_stamp, power, voltage 
                        from ts where year={year} and month={month} and is_pv=True and voltage > 0 and voltage < 300 and {split_cons}) as ts1
                inner join (select distinct site_id, circuit_id, circuit_polarity, ac_capacity_kw from meta_up23c where is_pv = True) as m on ts1.circuit_id = m.circuit_id
                group by site_id, t_stamp
                having avg(voltage) > 253),
                pq as (
                    select d.site_id, d.t_stamp, P_kW, d.V, ac_capacity_kw, vv.uncurtailed_P, vv.V as V_ghi,
                    (case when d.V < {v1} then ac_capacity_kw 
                    when d.V > {v2} then .2 * ac_capacity_kw
                    else ((ac_capacity_kw - .2 * ac_capacity_kw) / ({v1} - {v2}) * (d.V - {v2}) + .2 * ac_capacity_kw) end) + 0.04*ac_capacity_kw as max_P_volt_watt
                from data d left join voltwatt_uncurtailedPV as vv on d.site_id = vv.site_id and d.t_stamp = vv.t_stamp
                ),
                pq2 as (select site_id, t_stamp, day(t_stamp) as day, month(t_stamp) as month, year(t_stamp) as year, P_kW, uncurtailed_P, V, V_ghi, ac_capacity_kw, max_P_volt_watt,
                greatest(0, P_kW - max_P_volt_watt) as nonconformance_voltwatt,
                case when uncurtailed_P > max_P_volt_watt then greatest(0, P_kW - max_P_volt_watt) else null end as nonconformance_voltwattghi
                from pq
                )
                select * from pq2
                where  uncurtailed_P < max_P_volt_watt and nonconformance_voltwatt = 0 and V > 253
                """)
    # sleep(random.randint(1, 10))  # add some randomness to avoid overwhelming trino with simultaneous queries
    print(
        f"Completed year={year}, month={month},  {split_cons.replace('system.bucket(postcode, 16)', 'bucket')}"
    )
    return df


tasks = [
    (year, month, split_cons)
    for year in (2024,)
    for month in range(1, 2)
    for split_cons in [
        "system.bucket(postcode, 16) <= 7",
        "system.bucket(postcode, 16) > 7",
    ]
]
df = trino_parallel(run_func, tasks, num_workers=1)
df

Completed year=2024, month=1,  bucket <= 7
Completed year=2024, month=1,  bucket > 7
Combining results from all tasks.


,site_id,t_stamp,day,month,year,P_kW,uncurtailed_P,V,V_ghi,ac_capacity_kw,max_P_volt_watt,nonconformance_voltwatt,nonconformance_voltwattghi
0,45544685,2024-01-25 04:30:00,25,1,2024,5.372553,5.372553,254.60,254.60,8.0,6.857143,0.0,None
1,45544685,2024-01-23 04:30:00,23,1,2024,5.565290,5.748530,253.85,253.85,8.0,7.542857,0.0,None
2,45544685,2024-01-20 04:30:00,20,1,2024,5.345453,5.345453,254.80,254.80,8.0,6.674286,0.0,None
3,45544685,2024-01-12 04:30:00,12,1,2024,2.057487,4.814015,254.35,254.35,8.0,7.085714,0.0,None
4,45544685,2024-01-08 04:30:00,8,1,2024,5.075600,5.075600,253.45,253.45,8.0,7.908571,0.0,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
81961,1371025905,2024-01-19 03:45:00,19,1,2024,6.827867,6.827867,253.45,253.45,8.0,7.908571,0.0,None
81962,1371025905,2024-01-06 03:45:00,6,1,2024,6.606927,7.399657,253.90,253.90,8.0,7.497143,0.0,None
81963,1371025905,2024-01-09 02:35:00,9,1,2024,6.634093,7.451106,253.25,253.25,8.0,8.091429,0.0,None
81964,1371025905,2024-01-05 02:35:00,5,1,2024,6.924760,6.924760,253.20,253.20,8.0,8.137143,0.0,None


In [6]:
df = iceberg_sql(
    """select * from all_uncurtailedPV where year=2024 and month=1 and site_id=45544685 and t_stamp = timestamp '2024-01-25 04:30:00'"""
)
df["t_stamp"] = (
    pd.to_datetime(df["t_stamp"])
    .dt.tz_localize("UTC")
    .dt.tz_convert(pytz.FixedOffset(600))
)
df

,site_id,t_stamp,year,month,uncurtailed_p,p_kw,ghi,n_train
0,45544685,2024-01-25 14:30:00+10:00,2024,1,5.372553,5.372553,0.761238,139


In [5]:
def get_max_P(V, Srated=1, v1=253, v2=260):
    if V is None:
        return None
    if V < v1:
        return Srated
    elif V > v2:
        return 0.2 * Srated
    else:
        m = (Srated - 0.2 * Srated) / (v1 - v2)
        P = m * (V - v2) + 0.2 * Srated
        return P

In [13]:
v1 = 253
v2 = 260
iceberg_sql(f"""
                    with data as (
                        select site_id, t_stamp, sum(power*circuit_polarity/1000) as P_kW, 
                                avg(voltage) as V, ac_capacity_kw
                    from 
                    (select circuit_id, t_stamp, power, voltage 
                    from ts where year=2024 and month=1 and is_pv=True and voltage > 0 and voltage < 300) as ts1
                    inner join meta_up23c as m on ts1.circuit_id = m.circuit_id
                    group by site_id, t_stamp, ac_capacity_kw
                    having avg(voltage) > 253),
                    pq as (
                        select site_id, t_stamp, P_kW, V, ac_capacity_kw,
                    (case when V < {v1} then ac_capacity_kw 
                    when V > {v2} then .2 * ac_capacity_kw
                    else ((ac_capacity_kw - .2 * ac_capacity_kw) / ({v1} - {v2}) * (V - {v2}) + .2 * ac_capacity_kw) end) + 0.04*ac_capacity_kw as max_P_volt_watt
                    from data
                    ),
                    pq2 as (select site_id, t_stamp, day(t_stamp) as day, month(t_stamp) as month, year(t_stamp) as year, P_kW, V, ac_capacity_kw, max_P_volt_watt,
                    greatest(0, P_kW - max_P_volt_watt) as nonconformance_voltwatt 
                    from pq
                    )
                    
                    select site_id, day, month, year, 
                    sum(nonconformance_voltwatt) as nonconformance_voltwatt_sum,
                    sum(case when nonconformance_voltwatt > 0 then 1 else 0 end) as nonconformance_voltwatt_count,
                    count(nonconformance_voltwatt) as total_count
                    from pq2
                    group by site_id, day, month, year
                    order by nonconformance_voltwatt_sum desc
                    limit 10""")

,site_id,day,month,year,nonconformance_voltwatt_sum,nonconformance_voltwatt_count,total_count
0,1124511829,19,1,2024,544.442182,58,89
1,465008538,18,1,2024,435.721716,92,180
2,465008538,19,1,2024,405.846543,79,146
3,755163749,22,1,2024,394.168985,68,84
4,1124511829,13,1,2024,370.185861,56,92
5,755163749,11,1,2024,352.225633,71,93
6,465008538,11,1,2024,329.451779,77,229
7,814602644,4,1,2024,327.273590,63,77
8,814602644,3,1,2024,314.986680,64,74
9,755163749,31,1,2024,314.239765,56,62


In [7]:
iceberg_sql(
    "select circuit_id, circuit_polarity from meta_up23c where circuit_id = 467634"
)

OperationalError: (trino.exceptions.TrinoConnectionError) failed to execute: HTTPConnectionPool(host='trino.ciccada', port=8080): Max retries exceeded with url: /v1/statement (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x71f59b5ed0a0>: Failed to resolve 'trino.ciccada' ([Errno -2] Name or service not known)"))
[SQL: SET SESSION task_concurrency = 1]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [65]:
iceberg_sql("select * from meta_up23c limit 2")

,site_id,state,postcode,longitude,latitude,dnsp_name,dc_capacity_kw,ac_capacity_kw,export_limit_kw,monitoring_start,...,circuit_type,is_pv,min_time,max_time,v_95,v_05,m_id,n_long,n_lat,distance_km
0,1944472430,NSW,2502,150.85,-34.47,Endeavour,5.18,5.0,3.0,2021-08-09,...,pv_site_net,True,2024-07-23 04:45:00,2024-07-24 07:05:00,244.3,238.65,M43,150.84,-34.46,1.569777
1,1944472430,NSW,2502,150.85,-34.47,Endeavour,5.18,5.0,3.0,2021-08-09,...,ac_load_net,False,2024-07-23 04:45:00,2024-07-24 07:05:00,244.3,238.65,M43,150.84,-34.46,1.569777


In [18]:
iceberg_sql("""
            with data as (
                select ts.circuit_id, t_stamp, power as P_kW, 
                       energy_reactive*circuit_polarity/1000*12 as Q_kvar, voltage as V
            from ts left join meta_up23c as m on ts.circuit_id = m.circuit_id
            where year=2024 and month=1 and ts.is_pv=True and ts.circuit_id = 467634)
            select * from data order by t_stamp limit 10
            """)

,circuit_id,t_stamp,P_kW,Q_kvar,V
0,467634,2024-01-01 00:00:00,4473.9233,None,241.45
1,467634,2024-01-01 00:05:00,4523.3200,None,241.25
2,467634,2024-01-01 00:10:00,4535.8167,None,241.65
3,467634,2024-01-01 00:15:00,4589.9300,None,242.35
4,467634,2024-01-01 00:20:00,4619.6267,None,241.65
5,467634,2024-01-01 00:25:00,4716.0133,None,241.90
6,467634,2024-01-01 00:30:00,4767.7167,None,242.15
7,467634,2024-01-01 00:35:00,4758.5533,None,242.00
8,467634,2024-01-01 00:40:00,4803.2267,None,242.10
9,467634,2024-01-01 00:45:00,4833.1000,None,243.00


In [3]:
iceberg_sql("""select site_id, circuit_id from meta_up23c where is_pv = True and site_id = 616600992
            """)

,site_id,circuit_id
0,616600992,558200
1,616600992,558199
2,616600992,558198
